# Módulo DS5 · Redes y grafos

**Caso:** el club de kárate de Zachary (1977) es la red social más estudiada de la historia. Durante el estudio, un conflicto entre el instructor (nodo 0) y el presidente (nodo 33) partió el club en dos. ¿Podríamos haberlo anticipado solo mirando quién se relacionaba con quién?

Las mismas técnicas se usan para detectar fraude (redes de transacciones), encontrar influencers, analizar cadenas de suministro o recomendar contactos.

Objetivos:
1. Representar datos como grafos y calcular métricas básicas.
2. Medir la **centralidad** de los nodos con distintos criterios.
3. Entender y aplicar **PageRank**.
4. Detectar **comunidades** (modularidad y clustering espectral).
5. Predecir enlaces futuros.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import networkx as nx
from networkx.algorithms import community
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score

G = nx.karate_club_graph()
pos = nx.spring_layout(G, seed=3)
faccion_real = np.array([0 if G.nodes[n]["club"] == "Mr. Hi" else 1 for n in G])

def dibujar(G, colores=None, tamanos=300, titulo="", ax=None, cmap="coolwarm"):
    ax = ax or plt.gca()
    nx.draw_networkx(G, pos, node_color=colores if colores is not None else "lightblue", node_size=tamanos,
                     cmap=cmap, font_size=8, edge_color="lightgray", ax=ax)
    ax.set_title(titulo); ax.axis("off")

plt.figure(figsize=(8, 6)); dibujar(G, faccion_real, titulo="Club de kárate: facción real tras la ruptura"); plt.show()

## 1. Métricas básicas

In [ ]:
print(f"Nodos: {G.number_of_nodes()} | Aristas: {G.number_of_edges()}")
print(f"Densidad: {nx.density(G):.3f}")
print(f"Grado medio: {np.mean([d for _, d in G.degree()]):.2f}")
print(f"Distancia media entre dos miembros: {nx.average_shortest_path_length(G):.2f}")
print(f"Coeficiente de agrupamiento: {nx.average_clustering(G):.2f}")

grados = pd.Series(dict(G.degree()))
grados.value_counts().sort_index().plot.bar(title="Distribución de grados"); plt.xlabel("grado"); plt.show()

## 2. Centralidad: ¿quién es importante?

| Medida | Idea | Uso típico |
|---|---|---|
| Grado | nº de conexiones | popularidad |
| Intermediación | cuántos caminos más cortos pasan por el nodo | "puentes", cuellos de botella |
| Cercanía | distancia media al resto | rapidez para difundir información |
| Autovector | conectado con nodos a su vez importantes | influencia |
| PageRank | paseo aleatorio con saltos | ranking de webs, riesgo en redes |

In [ ]:
centralidades = pd.DataFrame({
    "grado": nx.degree_centrality(G),
    "intermediación": nx.betweenness_centrality(G),
    "cercanía": nx.closeness_centrality(G),
    "autovector": nx.eigenvector_centrality(G),
    "pagerank": nx.pagerank(G),
})
centralidades.sort_values("intermediación", ascending=False).head(8).round(3)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
dibujar(G, list(centralidades["grado"]), 3000 * centralidades["grado"], "Tamaño = grado", axes[0], "viridis")
dibujar(G, list(centralidades["intermediación"]), 3000 * centralidades["intermediación"] + 50, "Tamaño = intermediación", axes[1], "viridis")
plt.show()

## 3. PageRank desde cero
Un "paseante" salta de nodo en nodo siguiendo aristas; con probabilidad $1-d$ se teletransporta a un nodo al azar. PageRank es la fracción de tiempo que pasa en cada nodo. Se calcula por **iteración de potencias**:

In [ ]:
def pagerank(G, d=0.85, iters=100):
    A = nx.to_numpy_array(G, nodelist=sorted(G))
    M = A / A.sum(axis=0, keepdims=True)            # matriz de transición por columnas
    n = len(A)
    r = np.full(n, 1 / n)
    for _ in range(iters):
        r = (1 - d) / n + d * M @ r
    return r

pr_propio = pagerank(G)
print("Diferencia máxima con networkx:", np.abs(pr_propio - centralidades["pagerank"].values).max().round(6))

## 4. Detección de comunidades
### Optimización de modularidad (Louvain)
La **modularidad** compara la densidad de aristas dentro de los grupos con la esperada al azar.

In [ ]:
comunidades = community.louvain_communities(G, seed=0)
etiqueta_louvain = np.zeros(len(G), dtype=int)
for c, nodos in enumerate(comunidades):
    etiqueta_louvain[list(nodos)] = c
print(f"Louvain: {len(comunidades)} comunidades | modularidad = {community.modularity(G, comunidades):.3f}")

### Clustering espectral
Los autovectores de la **matriz laplaciana** $L = D - A$ asociados a los autovalores más pequeños codifican la estructura de grupos. El segundo (vector de Fiedler) da la mejor partición en dos.

In [ ]:
L = nx.laplacian_matrix(G, nodelist=sorted(G)).toarray().astype(float)
autovalores, autovectores = np.linalg.eigh(L)
fiedler = autovectores[:, 1]
etiqueta_espectral = (fiedler > 0).astype(int)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
dibujar(G, faccion_real, titulo="Facción real", ax=axes[0])
dibujar(G, etiqueta_louvain, titulo=f"Louvain (ARI vs real = {adjusted_rand_score(faccion_real, etiqueta_louvain):.2f})", ax=axes[1], cmap="tab10")
dibujar(G, etiqueta_espectral, titulo=f"Espectral 2 grupos (ARI = {adjusted_rand_score(faccion_real, etiqueta_espectral):.2f})", ax=axes[2])
plt.show()
print(f"Aciertos del corte espectral: {max((etiqueta_espectral == faccion_real).mean(), (etiqueta_espectral != faccion_real).mean()):.0%} de los miembros")

Solo con la estructura de amistades, el corte espectral predice la facción de casi todos los miembros. Louvain encuentra subgrupos más finos dentro de cada facción.

## 5. Predicción de enlaces
¿Qué pares de personas que **no** se conocen tienen más probabilidad de hacerse amigos? Una heurística clásica: el **índice de Adamic-Adar** (amigos en común, dando más peso a los amigos comunes con pocos contactos). Lo evaluamos ocultando el 10 % de las aristas.

In [ ]:
rng = np.random.default_rng(0)
aristas = list(G.edges())
ocultas = [aristas[i] for i in rng.choice(len(aristas), len(aristas) // 10, replace=False)]
G_train = G.copy(); G_train.remove_edges_from(ocultas)

no_aristas = list(nx.non_edges(G_train))
puntuaciones = sorted(nx.adamic_adar_index(G_train, no_aristas), key=lambda t: -t[2])
top = {(min(u, v), max(u, v)) for u, v, _ in puntuaciones[:len(ocultas)]}
ocultas_set = {(min(u, v), max(u, v)) for u, v in ocultas}
print(f"Aristas ocultas: {len(ocultas)} | recuperadas en el top-{len(ocultas)}: {len(top & ocultas_set)}")
print(f"Acierto esperado al azar: {len(ocultas) ** 2 / len(no_aristas):.2f}")

## Ejercicios

1. Quieres difundir un mensaje en el club contactando solo a **3 personas**. ¿A quiénes eliges y con qué criterio de centralidad? Simula la difusión (cada persona informada contagia a cada vecino con probabilidad 0,3 durante 5 rondas) y compara tu elección con 3 personas al azar.
2. Elimina el nodo con mayor intermediación. ¿Qué pasa con la distancia media y con el número de componentes conexas? ¿Qué implica para una cadena de suministro?
3. Compara Adamic-Adar con "número de vecinos comunes" y con `nx.jaccard_coefficient`. Repite con 20 semillas distintas y compara las medias.
4. Usa `nx.les_miserables_graph()` (personajes de *Los miserables*) y responde: ¿quién es el personaje más central según cada medida? ¿Coinciden?

In [ ]:
# TU CÓDIGO AQUÍ